# Readers who liked the books you liked also read...

Unit 3 asked the first reader-dependent question: *given the books you have read, which other books use the same words?* That is a **content** path -- it reads the catalog text and never looks at anyone else. It beat popularity by about 1.5x, a solid win, but it has a blind spot it can never close: two books with no words in common that the *same people* keep reading together are invisible to it.

This unit asks a different question, the one that gave recommendation its name: **readers who liked the books you liked also read...** -- which books? We never open the catalog text. We read only the interaction log -- *who read what* -- and let the crowd of readers draw the lines between books. This is **collaborative filtering** (CF): the first path whose signal is other readers rather than the item's own content.

We build it from scratch -- a co-occurrence count, then an item-item **cosine** similarity, then **k-nearest-neighbour** retrieval over that similarity -- reveal the `bookrec` implementation, and score it on the Unit 1 scoreboard. It is the first path to beat **both** popularity (~0.108) and the Unit-3 content/lexical path (~0.158): a decisive ~2.3x over popularity and ~1.6x over lexical -- the widest margin and the first real personalization win of the book. We will also be honest about what it *cannot* reach: about 41% of the catalog has no co-readership at all, a coverage ceiling that bridges to the cold-start blending of Unit 6.

## Load the catalog and the interaction log

The Unit 1/2/3 substrate is unchanged. `bookrec.generated_dir()` locates the seeded data slice (run the recsys generators first if it complains), `load_catalog` gives the `dict[int, Book]`, and the interaction log is the gzip-ed CSV with one row per reader-item event: `reader_id, item_id, split, label`. Collaborative filtering needs *none* of the catalog text -- only the log -- but we load the catalog so we can read back the genres of what the path recommends and watch it ignore them.

In [ ]:
import json

import bookrec
import numpy as np
import pandas as pd

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
interactions = pd.read_csv(interactions_path)

print(f"catalog: {len(catalog_ids)} books; interaction log: {len(interactions)} rows")
print("columns:", list(interactions.columns))

## 1. Co-occurrence and implicit feedback

Collaborative filtering rests on a single, almost childish observation: **two books are similar when the same readers engaged both.** No one has to describe the books; the readers do it for us by the company they keep. To turn that into arithmetic we first have to be precise about what a row of the log *means*.

### Implicit vs explicit feedback

Some systems have **explicit** feedback -- a reader types a rating, 1 to 5 stars, and a low rating is a genuine *negative*: "I read this and disliked it." Our log, like most real logs (clicks, plays, reads), is **implicit**: a reader either engaged a book or they did not. A read is a clear **positive**. But there is no explicit negative -- a book you never read might be one you would have hated, or one you simply never saw. Implicit feedback is **positive-only and sparse**: we observe a handful of likes per reader and nothing trustworthy about the vast rest of the catalog.

Our generated log makes this concrete, and honest, with a `label` column:

- `label == 1` -- a **positive**: the reader engaged the book. These are the rows we have filtered to since Unit 1.
- `label == 0` -- an **exposure-sampled negative**: the reader was *shown* the book (it entered their exposure) and did **not** engage it. It stands in for the unobserved "did not like," but it is a *sample*, not a true dislike -- the reader might simply not have gotten to it.

Count both in the train split:

In [ ]:
train = interactions[interactions["split"] == "train"]
n_pos = int((train["label"] == 1).sum())
n_neg = int((train["label"] == 0).sum())

print(f"train rows: {len(train)}")
print(f"  label==1 positives        (reader engaged the book): {n_pos}")
print(f"  label==0 sampled negatives (exposed, not engaged)   : {n_neg}")
print(f"positives are {n_pos / len(train):.1%} of the train log")

So the train log holds **5303 positives** against **10457 exposure-sampled negatives** -- more "did not engage" rows than "engaged" rows. What does Unit 4's co-occurrence path do with the negatives? **Nothing.** The similarity is built from **positives only** (`split == "train"` and `label == 1`): two books are neighbours because readers *liked* both, and a `label == 0` row carries no co-*liking* signal. The path then ranks the **full catalog** -- it never samples negatives to score against.

That is a deliberate, honest choice, not an oversight. The `label == 0` rows are genuinely useful, but for a *different* job: training a model to tell positives from negatives. That is exactly what **Unit 5 (matrix factorization)** and **Unit 8 (the two-tower model)** do -- they *use* the sampled negatives as training signal. Unit 4 needs no such thing: counting co-occurrence among positives is already enough to beat every path so far.

## 2. The item-item cosine co-occurrence similarity, by hand

Start with the raw material: a binary **reader x item incidence** matrix $\mathbf{R}$ built from the train positives, where $\mathbf{R}_{r,i} = 1$ exactly when reader $r$ has a train positive for item $i$, and $0$ otherwise. Each **column** $\mathbf{R}_{:,i}$ is the set of readers who liked book $i$; each **row** is one reader's history.

Two books $i$ and $j$ co-occur for every reader who liked both -- that is the dot product of their columns, $\mathbf{R}_{:,i} \cdot \mathbf{R}_{:,j}$, the plain count of shared readers. Raw co-occurrence has popularity's disease, though: a book everyone reads co-occurs with everything. So we normalize by each column's length, exactly the **cosine** of Unit 3 but now between two *item* columns instead of two keyword vectors:

$$\mathrm{sim}(i, j) = \frac{\mathbf{R}_{:,i} \cdot \mathbf{R}_{:,j}}{\lVert \mathbf{R}_{:,i} \rVert \, \lVert \mathbf{R}_{:,j} \rVert} = \frac{\lvert \text{readers who liked both } i \text{ and } j \rvert}{\sqrt{\lvert \text{readers of } i \rvert}\;\sqrt{\lvert \text{readers of } j \rvert}}$$

The cosine is $1$ when two books have exactly the same readers, $0$ when they share none, and it discounts a popular book's reach because its large norm sits in the denominator. We zero the **diagonal** so a book is never its own neighbour. One subtlety: a book nobody liked is an all-zero column with norm $0$; dividing would give $0/0$, so we clamp the denominator to a tiny floor and that book gets an all-zero similarity row -- harmless now, but the seed of the coverage ceiling in section 3.

One bookkeeping note before we build. The **5303 train positives** are *rows*, but a handful of `(reader, item)` positives repeat across rows, so they collapse to **4694 distinct `(reader, item)` pairs**. Co-occurrence reads the incidence, which marks each reader-book link exactly once, so it counts **pairs** (4694) -- whereas popularity back in Unit 2 counted **rows**. That is why the incidence below reports 4694 non-zero entries, not 5303.

Build the incidence from the train positives:

In [ ]:
train_positive = train[train["label"] == 1]
pairs = set(zip(train_positive["reader_id"].astype(int), train_positive["item_id"].astype(int)))

# Index readers (from the positives) and items (the whole catalog is the item universe).
reader_ids = sorted({reader for reader, _ in pairs})
reader_of = {reader: row for row, reader in enumerate(reader_ids)}
col_of = {item_id: col for col, item_id in enumerate(catalog_ids)}

incidence = np.zeros((len(reader_ids), len(catalog_ids)))
for reader, item_id in pairs:
    incidence[reader_of[reader], col_of[item_id]] = 1.0

assert incidence.sum() == len(pairs)
print(f"incidence matrix {incidence.shape}  (readers x items)")
print(f"non-zero entries (reader-liked-book pairs): {int(incidence.sum())}")

Now the cosine, directly from the definition: the numerator is the item-item dot product $\mathbf{R}^{\top}\mathbf{R}$ (every shared-reader count at once), and the denominator is the outer product of the column norms, clamped away from zero.

In [ ]:
norms = np.sqrt((incidence ** 2).sum(axis=0))                 # ||column_i|| = sqrt(#readers of i)
sim_by_hand = (incidence.T @ incidence) / np.maximum(np.outer(norms, norms), 1e-9)
np.fill_diagonal(sim_by_hand, 0.0)                            # a book is not its own neighbour

# Read one cell back as the plain-English count it encodes.
i, j = col_of[307], col_of[1066]                              # two of reader 3's books
co_readers = int((incidence[:, i] * incidence[:, j]).sum())
print(f"books 307 and 1066 share {co_readers} reader(s); cosine sim = {sim_by_hand[i, j]:.3f}")
print(f"similarity matrix {sim_by_hand.shape}, range {sim_by_hand.min():.3f} .. {sim_by_hand.max():.3f}")

### Reveal `bookrec.item_item_cosine`

The package ships exactly this computation. `bookrec.item_item_cosine(incidence)` takes the same binary reader x item matrix, returns the cosine item-item similarity with a zeroed diagonal and the same tiny-norm clamp, and agrees with our by-hand matrix to floating-point precision.

In [ ]:
sim_lib = bookrec.item_item_cosine(incidence)
assert np.allclose(sim_lib, sim_by_hand)
assert np.allclose(np.diag(sim_lib), 0.0)                     # diagonal zeroed
print("by-hand cosine co-occurrence == bookrec.item_item_cosine")

## 3. k-nearest-neighbour retrieval: score what your history points to

The similarity matrix knows which books go together. To recommend *for a reader*, we use their history as a query: a candidate book is good to the extent it is similar to the books the reader has already liked. The simplest and, on this data, the strongest rule is to **sum the similarity** of a candidate to every book in the reader's `seen` set:

$$\mathrm{score}(c) = \sum_{s \,\in\, \mathrm{seen}} \mathrm{sim}(c, s)$$

This is **k-nearest-neighbour (k-NN) retrieval** over the item-similarity matrix: the reader's recommendations are the neighbours of the books they already like. We then drop the books they have already read -- a re-read is not a recommendation -- which is what `rank(exclude=seen)` is for. (The *k* here is the number of recommendations to return; do not confuse it with the optional `n_neighbors` cap we meet below.)

Take reader 3 (the reader Unit 3 followed), whose `seen` set is their train positives.

In [ ]:
seen_by = (
    train_positive.groupby("reader_id")["item_id"]
    .apply(lambda items: {int(i) for i in items})
    .to_dict()
)

reader_id = 3
seen = seen_by[reader_id]
seen_cols = [col_of[item_id] for item_id in sorted(seen)]
scores = sim_by_hand[:, seen_cols].sum(axis=1)                # summed similarity to the history

candidates = [
    bookrec.Candidate(item_id, float(scores[col]), "cooc-by-hand")
    for col, item_id in enumerate(catalog_ids)
]
top5 = bookrec.rank(candidates, n=5, exclude=seen)

print(f"reader {reader_id} has read {sorted(seen)}")
print(f"{'rec':>6} {'score':>6}  genres and this book's similarity to each read book")
for c in top5:
    col = col_of[c.item_id]
    neighbours = " ".join(f"{s}:{sim_by_hand[col_of[s], col]:.2f}" for s in sorted(seen))
    print(f"{c.item_id:>6} {c.score:>6.3f}  [{catalog[c.item_id].fields['genres']}]  {neighbours}")

Look at what came back and, more tellingly, what did **not** steer it: the genres. Unit 3's lexical path recommended reader 3 five *poetry* books because their words matched a poetry-heavy history. Reader 3's own history spans science, sci-fi, poetry, mystery, history, biography and adventure -- yet the top recommendation, **book 976, is `fantasy;romance`**, two genres reader 3 has never touched. It still tops the list, and it earns its place from the `sim` columns on the right: it is a neighbour of a *specific* book reader 3 liked, because other readers engaged both. That is the collaborative signal: the recommendation rides on *shared readership*, not shared content. The two signals are complementary, which is why Unit 6 will blend them.

### Reveal `ItemItemRetrievalPath`, register it, and score it

The package packages the by-hand pieces as a standard retrieval path. `bookrec.ItemItemRetrievalPath()` fits the item-item cosine similarity from the **train positives** of the interaction log (`fit` duck-types the row-mapping log exactly like `PopularityRetrievalPath` -- no pandas inside the package -- and honours `catalog=` as the item universe). `retrieve(query, context, k)` scores each candidate by the summed similarity to `context["seen"]`, excludes the seen books, and returns the top `k` as `Candidate`s. A reader with no history gets `[]`, the honest answer from a path that depends on history. It fits the *same* `fit(interactions, catalog=None)` signature as every other path, so it drops straight into the registry and the scoreboard.

In [ ]:
rows = interactions.to_dict("records")
cf = bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)

# The shipped path reproduces our by-hand ranking for reader 3.
shipped = cf.retrieve(query=reader_id, context={"seen": seen}, k=5)
assert [c.item_id for c in shipped] == [c.item_id for c in top5]

# A reader with no history has nothing to be similar to.
assert cf.retrieve(query=999_999, context={"seen": set()}, k=5) == []
print("shipped ItemItemRetrievalPath reproduces the by-hand top-5; empty history -> []")

Now the scoreboard, under the Unit 1 rules unchanged: hit-rate@10 over positive `val` rows, each reader's train-positive items removed from both the recommendations and the relevance set, `cold_readers` from `cold_partitions.json` excluded, the random floor seeded at 0. We line CF up against the random floor, popularity (Unit 2) and lexical (Unit 3).

In [ ]:
cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])

keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=0)
popularity = bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)

registry = bookrec.PathRegistry()
for path in (random_floor, popularity, lexical, cf):
    registry.register(path)


def score(path):
    return bookrec.run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
    )


boards = {
    "random": score(random_floor),
    "popularity": score(popularity),
    "lexical": score(lexical),
    "item-item CF": score(cf),
}

floor = boards["random"].hit_rate_at_k
print(f"readers scored: {boards['item-item CF'].readers}")
print(f"{'path':>14} {'hit@10':>8} {'recall@10':>10} {'vs floor':>9}")
for label, board in boards.items():
    print(f"{label:>14} {board.hit_rate_at_k:>8.4f} {board.recall_at_k:>10.4f} "
          f"{board.hit_rate_at_k / floor:>8.1f}x")

cf_hit = boards["item-item CF"].hit_rate_at_k
print(f"\nCF vs popularity: {cf_hit / boards['popularity'].hit_rate_at_k:.2f}x   "
      f"CF vs lexical: {cf_hit / boards['lexical'].hit_rate_at_k:.2f}x")

### Read the scoreboard honestly

Item-item CF scores about **0.252** hit@10 -- roughly **21x** the random floor, **2.3x** popularity (~0.108) and **1.6x** the lexical path (~0.158). This is the headline of Part 1. Unit 3's lexical path already modestly beat popularity (~1.46x), so CF is not the *first* path to beat popularity -- but it is the first **collaborative** path, and the first to beat **both** popularity and the content path, by the widest margin we have seen. Reading *who read what* carries more signal here than reading either the crowd's totals or the books' own words, because the generator imprinted each reader's latent taste into who-reads-what, and co-occurrence recovers it.

Why does it work? Because the summed-similarity score is, in effect, a vote: every book you liked votes for its neighbours, and the books with the most votes from your particular history rise. That is personalization in its simplest honest form.

### Two honest limits

**The neighbourhood cap does not help here.** The name "k-nearest-neighbour" suggests we should keep only each book's top few neighbours and zero the rest -- a smaller, cleaner neighbourhood. `ItemItemRetrievalPath` offers that as an optional `n_neighbors` cap, and it is worth *measuring* rather than assuming. On this data it **degrades** the score: with so few positives per book, the handful of neighbours a cap keeps throws away real co-occurrence signal.

In [ ]:
capped = bookrec.ItemItemRetrievalPath(n_neighbors=10).fit(rows, catalog=catalog_ids)
capped_hit = score(capped).hit_rate_at_k
print(f"uncapped item-item CF : {cf_hit:.4f}  (default)")
print(f"n_neighbors=10 cap    : {capped_hit:.4f}  <- lower than uncapped, and below lexical")
print("so the default is UNCAPPED: a tighter neighbourhood discards signal on this sparse log")

**The coverage ceiling.** A collaborative path can only recommend a book that *someone* co-read -- a book with no train positives has an all-zero similarity row and can never be a neighbour of anything. Count how many books that leaves unreachable.

In [ ]:
reachable = int((sim_lib.sum(axis=1) > 0).sum())
unreachable = len(catalog_ids) - reachable
print(f"books reachable by CF (have co-readership): {reachable}")
print(f"books CF can NEVER recommend (no train positives): {unreachable} "
      f"({unreachable / len(catalog_ids):.1%} of the catalog)")

About **41%** of the catalog (818 of 2000 books) is invisible to CF. This is the mirror image of the lexical path's strength: lexical can recommend a brand-new book from its words alone, with no readers at all, while CF cannot touch a book until the crowd has. Neither path covers the whole catalog by itself -- which is precisely the argument for **blending** them, the subject of Unit 6's cold-start work.

## Where this goes

- **Co-occurrence** turns the interaction log into similarity: two books are alike when the same readers like both. No catalog text, just the crowd -- the first **collaborative** signal.
- **Implicit feedback** means positives-only and sparse. The co-occurrence path uses the `label == 1` positives and ignores the `label == 0` sampled negatives; those negatives become training signal in Unit 5 (matrix factorization) and Unit 8 (the two-tower model), not here.
- **Item-item cosine** normalizes raw co-occurrence so popular books do not dominate; **k-NN retrieval** scores a candidate by the summed similarity to a reader's history. The uncapped neighbourhood is best on this data; a small `n_neighbors` cap *degrades* it.

`ItemItemRetrievalPath` is the first collaborative path in the registry, and the first to beat **both** popularity and the content/lexical path -- about 0.252 hit@10, ~2.3x popularity and ~1.6x lexical. It is also blind to 41% of the catalog that no one has co-read yet, exactly where the content path still shines.

### The low-rank bridge to matrix factorization

The co-occurrence **structure** underneath our similarity is the cosine-normalized **Gram matrix** $\mathbf{G} = \mathbf{D}^{-1/2}\,\mathbf{R}^{\top}\mathbf{R}\,\mathbf{D}^{-1/2}$ of the reader x item incidence $\mathbf{R}$ (with $\mathbf{D}$ the diagonal of column norms): every entry is a cosine-normalized shared-reader count, and the rank of $\mathbf{G}$ is at most $\mathrm{rank}(\mathbf{R}^{\top}\mathbf{R}) = \mathrm{rank}(\mathbf{R})$, which the sparse log already keeps small. The **shipped** similarity then **zeroes the diagonal** -- so a book is never its own neighbour -- a retrieval convenience that can *raise* the matrix's rank, so the low-rank claim is about $\mathbf{G}$, the co-occurrence structure, **not** the shipped zero-diagonal matrix. **Unit 5's matrix factorization** approximates that low-rank structure with a *deliberately smaller* latent rank -- the **low-rank view** -- factoring taste into a handful of latent dimensions instead of one axis per book. That low-rank approximation *generalizes* co-occurrence for the **warm** books -- those that already have interactions -- filling in plausible similarities that a raw pairwise count never literally observed. What it cannot do is conjure signal from nothing: an id-only factorization still has no factors for the 818 **cold** books with zero train positives, so the coverage ceiling survives Unit 5 untouched. Closing it takes two later moves -- **Unit 6** blends the collaborative and content candidates, and **Unit 9** gives items a feature-based cold start -- so the recommender stops having a blind spot at all.